# Stage 3 - Rerieval Pipeline

## Data
2_chunks.json
1_queries.json

## Install
```
pip install sentence_transformers numpy 
pip install tqdm
pip install chromadb
pip install openai
```

In [1]:
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]
metas = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks]

print("Разделов:", len(chunks))

Разделов: 202


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

def embed_passages(model, texts):
    """Эмбеддинги документов через multilingual-e5: префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=64)


# --- Считаем эмбеддинги корпуса (e5-base, 768 координат) и сохраняем ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")
corpus_vecs = np.ascontiguousarray(embed_passages(embedder, texts), dtype="float32")
np.save("tmp/kb_vecs.npy", corpus_vecs)   # Эмбеддинги — в файл, переиспользуем

print(f"Чанков: {len(texts)}, эмбеддинги: {corpus_vecs.shape}")
print("Пример метаданных:", metas[1])

/Users/german/work/yai_02/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 16:11:30.571000 1254 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6910.46it/s]


Чанков: 202, эмбеддинги: (202, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# --- Создаём локальную коллекцию Chroma ---
client = chromadb.PersistentClient(path="chroma_kb")     # Хранение на диске
try:
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон
except Exception:
    pass

col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col.add(
    ids=[str(i) for i in range(len(texts))],             # Уникальные id
    embeddings=corpus_vecs.tolist(),                     # Векторы чанков
    documents=texts,                                     # Сами тексты
    metadatas=metas,                                     # Метаданные
)
print("Записей в коллекции:", col.count())

Записей в коллекции: 202


In [114]:
# фильтрацию по метаданным на одном примере

# --- Семантический поиск с фильтром по категории ---
QUERY = "какая версия webOS или Tizen нужна для приложения на телевизоре"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=5,
  where={"category": "Устройства и качество"}
)                   # Фильтр по метаданным

print(f"\nЗапрос: {QUERY!r} | фильтр category='Устройства и качество'")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] [{m['category']}] {doc}")

# Для сравнения — тот же запрос без фильтра
res2 = col.query(
  query_embeddings=[q_vec.tolist()], 
  n_results=5
)
print("\nБез фильтра категории первых трёх:",
      [m["category"] for m in res2["metadatas"][0]])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7415.11it/s]



Запрос: 'какая версия webOS или Tizen нужна для приложения на телевизоре' | фильтр category='Устройства и качество'
  dist=0.129 [id: 161] [doc_id: 146] [title: На каких устройствах работает «Фильмоскоп»] [Устройства и качество] # На каких устройствах работает «Фильмоскоп» Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года), LG (webOS 4.0 и новее), телевизоры и приставки на Android TV 9 и новее. ТВ-приставки: Android TV, Apple TV (tvOS 15 и новее), приставки операторов-партнёров. Телефоны и планшеты: iOS 15 и новее, Android 9 и новее. Компьютеры: браузеры Chrome, Firefox, Safari и Яндекс Браузер актуальных версий — без установки приложения. Игровые консоли: PlayStation 5 и Xbox Series. На устройствах вне этого списка работа приложения не гарантируется, даже если его удалось установить.
  dist=0.139 [id: 95] [doc_id: 86] [title: Поддерживаемые устройства] [Устройства и качество] # Поддерживаемые устройства На каких устройствах работает "Фильмоскоп" Телевизоры Smart TV: Samsung (Ti

In [ ]:
# precision@K, recall@K и MRR

In [6]:
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.83

# load queries
with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve(query, col, embedder, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out


for qi, q in enumerate(queries):
  hits = retrieve(q["query"], col, embedder, k=5, threshold=THRESHOLD)
  print(f"\n[{qi}] {q['query']}")
  if not hits:
    print("    (ничего выше порога — пайплайн вернул пусто)")
  for cid, doc, sim, id in hits:
    print(f"    sim={sim:.3f} #{cid} {doc[:58]} #{id}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6504.17it/s]



[0] что означает ошибка P-102
    sim=0.885 #70 # Ошибка P-102 (архив) Внимание: статья устарела и хранитс #77
    sim=0.861 #85 # Ошибка P-102: превышен лимит просмотров Ошибка P-102 поя #94
    sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает св #198
    sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить #150
    sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает  #151

[1] какая версия webOS или Tizen нужна для приложения на телевизоре
    sim=0.871 #146 # На каких устройствах работает «Фильмоскоп» Телевизоры Sm #161
    sim=0.861 #86 # Поддерживаемые устройства На каких устройствах работает  #95
    sim=0.839 #139 Если вы не знаете, к какой платформе относится ваш телевиз #152
    sim=0.838 #129 Ошибка P-105 появляется, когда на устройстве установлена у #142
    sim=0.835 #172 Приложение на телевизоре обновляется само, если в магазине #191

[2] обязателен ли HDCP 2.2 для просмотра в 4K
    sim=0.874 #110 Что нужно 

In [116]:
# col, embedder, retrieve(), queries, chunks — из Задания 4

relevant = {qi: set(q["relevant_docs"]) for qi, q in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    hits = retrieve(q["query"], col, embedder, k=5, threshold=0.0)
    top_ids = [int(cid) for cid, _, _, id in hits]
    rel = relevant[qi]
    is_problem = (not top_ids) or (top_ids[0] not in rel)
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    hits = retrieve(q["query"], col, embedder, k=10, threshold=0.0)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for cid, doc, sim, id in hits:
        mark = "+" if int(cid) in rel else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}")

print
print("\nЗапрос [8] без порога:",
  [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.0)])
print("Запрос [8] с порогом 0.83:",
  [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.83)])

Проблемные запросы: [0, 3, 6, 7, 8, 10, 11, 12, 15, 18, 19, 20, 22, 23, 24, 28, 29, 30, 31, 33, 34, 35, 37, 38, 39, 40, 41, 45, 46, 47, 48, 49, 50, 51, 53, 55]

[0] что означает ошибка P-102  (релевантные id: [85])
      sim=0.885 #70 # Ошибка P-102 (архив) Внимание: статья устарела и хранится 
    + sim=0.861 #85 # Ошибка P-102: превышен лимит просмотров Ошибка P-102 появл
      sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает своб
      sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить с
      sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает за
      sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена уст
      sim=0.851 #51 Ошибка P-113 появляется, когда файл загрузки повреждён. Сооб
      sim=0.850 #90 Ошибка P-110 появляется, когда происходит сбой на стороне се
      sim=0.849 #94 Ошибка P-101 появляется, когда приложению не удаётся соедини
      sim=0.848 #23 Ошибка P-107 (браузер) Ошибка P-

In [10]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [11]:
from collections import defaultdict

def search(query, n=50):
    """Поиск по тексту: кодирует его как query: и возвращает индексы
    top-n документов по убыванию косинусного сходства."""
    ret = retrieve(q["query"], col, embedder, k=5, threshold=0.0)
    return ret


def expand(query, n=5):
    """Query expansion: n перефразировок запроса официальными словами базы."""
    text = generate([
        {"role": "system", "content": "Ты помощник поиска по базе знаний "
                                      "онлайн-кинотеатра."},
        {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
         f"официальными словами из базы знаний. Используй полные фразы (не одно "
         f"слово), синонимы и термины. По одной формулировке на строку, без "
         f"нумерации.\n\nЗапрос: {query}"}])
    return [l.strip() for l in text.splitlines() if l.strip()][:n]


def rrf(ranked_lists, k=60, top_k=5):
    """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[doc] += 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [doc for doc, _ in ranked[:top_k]]

def get_text(doc_id):
  return next((i for i in corpus if i.get("doc_id") == doc_id), None)

for qi in [22, 30, 33]:
    query = queries[qi]
    rel = set(queries[qi]["relevant_docs"])
    paras = expand(query)
    expanded = rrf([search(query)] + [search(p) for p in paras])
    mark = lambda i: "+" if i in rel else " "
    print(f"\n[{qi}] {query}")
    for p in paras:
      print("   ·", p)

    found = search(query)
    # print('>>> found: ', found)

    print("  обычный поиск top5:",
      [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in found[:5]])
    print("  query expansion top5:",
      [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in expanded[:5]])


[22] {'query': 'начал в метро с телефона, дома хочу досмотреть на большом экране с той же секунды', 'kind': 'semantic', 'relevant_docs': [111], 'gold': 'позиция синхронизируется автоматически, фильм появится в разделе «Продолжить просмотр» на главном экране'}
   · Как только пользователь возобновит просмотр фильма в метро с мобильного устройства, система автоматически синхронизирует прогресс и предложит продолжить просмотр на большом экране в разделе «Продолжить просмотр».
   · При возобновлении просмотра фильма с мобильного устройства в общественном транспорте, прогресс просмотра синхронизируется автоматически и доступен в разделе «Продолжить просмотр» на главном экране.
   · Если пользователь начал просмотр фильма на телефоне в метро, то при переходе на домашний экран фильм появится в разделе «Продолжить просмотр» с сохранением точного момента остановки.
   · Система автоматически синхронизирует прогресс просмотра фильма, начатого на мобильном устройстве в метро, и делает его доступ

In [12]:
def retrieve_hyde(query, answer, col, embedder, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out


def hyde(query, top_k=5):
    """HyDE: ищем по среднему векторов запроса и гипотетического ответа."""
    doc = generate([
        {"role": "system", "content": "Ты пишешь короткие ответы из базы знаний "
                                      "интернет-магазина."},
        {"role": "user", "content": f"Напиши короткий (1-2 предложения) фрагмент-"
         f"ответ из базы знаний на вопрос клиента официальным языком. Только "
         f"текст ответа.\n\nВопрос: {query}"}], max_tokens=110)
    hits = retrieve_hyde(query, doc, col, embedder, top_k)
    return doc, hits


for qi in [22, 30, 33]:
    query = queries[qi]["query"]
    rel = set(queries[qi]["relevant_docs"])
    doc, hits = hyde(query)
    mark = lambda i: "+" if i in rel else " "

    found = search(query)
    # print('>>> found: ', found)

    print(f"\n[{qi}] {query}")
    print("  гипотетический ответ:", doc)
    print("  обычный поиск top5:",
        [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in found[:5]])
    print("  HyDE top5:", [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in hits[:5]]) 


[22] начал в метро с телефона, дома хочу досмотреть на большом экране с той же секунды
  гипотетический ответ: Для просмотра контента, начатого на мобильном устройстве в метро, на большом экране дома необходимо использовать функцию «Продолжить просмотр» или «Синхронизация», которая автоматически сохраняет прогресс. Пожалуйста, войдите в свой аккаунт на компьютере или планшете, чтобы продолжить просмотр с того же места.
  обычный поиск top5: [' #78 Статья обновляется Раздел времен', ' #55 Раздел справки (архив) Страница ', ' #41 ticket_id;category;status;resolv', ' #93 Внимание: статья устарела и хран', ' #201 Заменить привязанную карту можно']
  HyDE top5: [' #26 # Какой тариф подойдёт именно ва', ' #159 Ограничения: **скачивать можно н', '+#122 Позиция просмотра сохраняется ав', ' #78 Статья обновляется Раздел времен', ' #124 # Тарифы «Фильмоскопа» Тариф выб']

[30] боюсь, что ребёнок случайно купит фильм с моей карты
  гипотетический ответ: Для предотвращения случайных покупок реком

In [ ]:
import numpy as np

def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

rows = {"Обычный поиск": [], "Query expansion": [], "HyDE": []}

for q in queries:
    query, rel = q["query"], set(q["relevant_docs"])

    # base = search(query)
    # base_docs = [t[0] for t in base]
    # rows["Обычный поиск"].append(recall_at_k(base_docs, rel))

    # exp = rrf([search(query)] + [search(p) for p in expand(query)])
    # exp_docs = [t[0] for t in exp]
    # rows["Query expansion"].append(recall_at_k(exp_docs, rel))

    _, hyd = hyde(query)
    hyd_docs = [t[0] for t in hyd]
    rows["HyDE"].append(recall_at_k(hyd_docs, rel))

n = len(queries)
print(f"{'Подход':<18}{'recall@5':>10}")
for name in rows:
  print(f"{name:<18}{np.mean(rows[name]):>10.3f}")

[70, 138, 129, 178, 137]
[146, 86, 139, 66, 172]
[110, 45, 17, 110, 152]
[125, 69, 75, 50, 28]
[178, 70, 138, 129, 90]
[49, 51, 90, 165, 138]
[88, 113, 25, 24, 71]
[92, 155, 155, 71, 52]
[17, 101, 45, 152, 110]
[133, 146, 86, 113, 150]
[63, 62, 123, 63, 140]
[28, 22, 71, 75, 52]
[33, 166, 9, 131, 62]
[144, 33, 71, 52, 67]
[139, 153, 66, 138, 71]
[101, 112, 177, 71, 96]
[112, 71, 52, 125, 90]
[125, 71, 129, 52, 176]
[114, 71, 61, 64, 115]
[108, 140, 147, 84, 63]
[71, 52, 55, 169, 9]
[71, 64, 68, 64, 52]
[24, 144, 111, 71, 113]
[71, 52, 144, 144, 94]
[71, 52, 127, 55, 74]
[71, 180, 52, 154, 39]
[71, 52, 20, 20, 24]
[34, 147, 42, 3, 108]


In [100]:
print(recall_at_k([1,2,3], set([2])))

1.0


# Stage 3 - Report
- transformation is needed: Query expansion